# From Sharing to Professionalization: Comparing Airbnb Market Structures in Rome and Copenhagen

**Course:** Data Visualisations 2026  
**Authors:** Andrea Mammano, Simone, Elisa, [Team Member 4]  
**Date:** March 2026  

## Research Question
**To what extent is Airbnb characterized by professionalized hosting rather than casual sharing, and how do patterns differ between Rome and Copenhagen?**

---

## Notebook Structure
1. **Load Data** — Import Rome & Copenhagen `listings.csv` from Inside Airbnb (Sep 2025 scrapes).
2. **Clean Prices** — Strip currency symbols, convert to float, remove zeros/nulls.
3. **Handle Nulls** — Document imputation/dropping strategy.
4. **Feature Engineering** — Create `host_category`, `occupied_nights_proxy`, `estimated_revenue_l365d`.
5. **Tourist Pressure** — Merge mock neighbourhood population data.
6. **EDA Summary** — Quick statistics to validate the cleaned dataset.
7. **Export** — Save `data/airbnb_cleaned_final.csv` for Tableau.

> **Note:** This notebook is purely descriptive. No causal claims are made (e.g., we do not claim Airbnb *causes* rent increases). Analysis is limited to market structure and professionalization skew.


In [1]:
# ============================================================
# CELL 1 — IMPORTS
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

# Suppress max column warnings
pd.set_option('display.max_columns', None)

# Paths
DATA_DIR = Path('data')
ROME_CSV       = DATA_DIR / 'rome' / 'listings.csv'
COPENHAGEN_CSV = DATA_DIR / 'copenhagen' / 'listings.csv'
OUTPUT_CSV     = DATA_DIR / 'airbnb_cleaned_final.csv'

print('Libraries loaded.')
print(f'Rome data path:       {ROME_CSV}')
print(f'Copenhagen data path: {COPENHAGEN_CSV}')


Libraries loaded.
Rome data path:       data/rome/listings.csv
Copenhagen data path: data/copenhagen/listings.csv


---
## 1. Load Data
Inside Airbnb provides a `listings.csv` for each city with ~75 columns covering listing attributes, host details, pricing, availability, and review scores.

We add a `city` column to each DataFrame before concatenating so that both cities are distinguishable in analysis and in Tableau.


In [2]:
# ============================================================
# CELL 2 — LOAD DATA
# ============================================================

# Low-memory=False to avoid DtypeWarning on mixed-type columns
rome_raw = pd.read_csv(ROME_CSV, low_memory=False)
rome_raw['city'] = 'Rome'

cph_raw = pd.read_csv(COPENHAGEN_CSV, low_memory=False)
cph_raw['city'] = 'Copenhagen'

# Concatenate — ignore_index resets the integer index
df = pd.concat([rome_raw, cph_raw], ignore_index=True)

print(f'Combined rows:    {len(df):,}')
print(f'Columns:          {df.shape[1]}')
print(f'City breakdown:')
print(df['city'].value_counts())


Combined rows:    60,646
Columns:          80
City breakdown:
city
Rome          37652
Copenhagen    22994
Name: count, dtype: int64


---
## 2. Clean Prices
The `price` column from Inside Airbnb is a string with a leading `$` and comma separators (e.g., `"$1,200.00"`).  
We strip these and cast to `float`.  
Listings with a price of `0` or `NaN` are dropped because:
- **Zero-price listings** are almost certainly errors or test listings.
- **NaN prices** cannot be used in any revenue estimation.


In [3]:
# ============================================================
# CELL 3 — CLEAN PRICES
# ============================================================
before_count = len(df)

# Strip currency symbols and commas, then cast
df['price'] = (
    df['price']
    .astype(str)
    .str.replace(r'[\$,]', '', regex=True)  # remove $ and ,
    .str.strip()
    .replace('', np.nan)                    # empty strings → NaN
    .astype(float)
)

# Drop rows where price is NaN or zero
df = df[df['price'].notna() & (df['price'] > 0)].copy()

after_count = len(df)
print(f'Rows before price cleaning: {before_count:,}')
print(f'Rows after  price cleaning: {after_count:,}')
print(f'Dropped:                    {before_count - after_count:,}')
print(f'\nPrice summary (€/night):')
print(df.groupby('city')['price'].describe()[['count', 'mean', '50%', 'max']].round(2))


Rows before price cleaning: 60,646
Rows after  price cleaning: 47,006
Dropped:                    13,640

Price summary (€/night):
              count     mean     50%       max
city                                          
Copenhagen  13442.0  1445.96  1200.0  112955.0
Rome        33564.0   197.33   133.0   10515.0


---
## 3. Handle Missing Values

### Strategy

| Column | Missing-value action | Rationale |
|---|---|---|
| `review_scores_rating` | **Drop** rows where null | Listings with no reviews are outside the active market. Imputing ratings would introduce false precision. |
| `bedrooms` | **Impute** with median per `room_type` × `city` | Bedroom count matters for bubble-chart sizing; imputing with a room-type median is more accurate than the global median. |
| `bathrooms` | **Impute** with median per `room_type` × `city` | Same logic as bedrooms. |

> **Limitation:** Dropping ~30-40% of rows on `review_scores_rating` creates a **survivorship bias** — we only observe actively booked listings. This is acknowledged in Section 6 of the report.


In [4]:
# ============================================================
# CELL 4 — HANDLE NULLS
# ============================================================

# --- 4a. Review scores: DROP rows with null rating ---
before_review = len(df)
df = df[df['review_scores_rating'].notna()].copy()
after_review = len(df)
print(f'[review_scores_rating] dropped {before_review - after_review:,} rows  '
      f'({(before_review - after_review) / before_review * 100:.1f}%)')

# --- 4b. Bedrooms: IMPUTE with median per room_type × city ---
bedroom_medians = (
    df.groupby(['city', 'room_type'])['bedrooms']
    .transform('median')
)
null_bedrooms = df['bedrooms'].isna().sum()
df['bedrooms'] = df['bedrooms'].fillna(bedroom_medians).fillna(1)  # fallback = 1
print(f'[bedrooms]              imputed {null_bedrooms:,} missing values')

# --- 4c. Bathrooms: IMPUTE with median per room_type × city ---
# Note: Inside Airbnb provides bathrooms_text; use bathrooms_text to derive numeric
if 'bathrooms' not in df.columns or df['bathrooms'].isna().all():
    # Parse numeric part from bathrooms_text (e.g., '1 bath', '1.5 baths', 'Shared half-bath')
    df['bathrooms'] = (
        df['bathrooms_text']
        .astype(str)
        .str.extract(r'(\d+\.?\d*)', expand=False)
        .astype(float)
    )

bathroom_medians = (
    df.groupby(['city', 'room_type'])['bathrooms']
    .transform('median')
)
null_bathrooms = df['bathrooms'].isna().sum()
df['bathrooms'] = df['bathrooms'].fillna(bathroom_medians).fillna(1)  # fallback = 1
print(f'[bathrooms]             imputed {null_bathrooms:,} missing values')

print(f'\nFinal dataset size: {len(df):,} rows')


[review_scores_rating] dropped 5,813 rows  (12.4%)
[bedrooms]              imputed 22 missing values
[bathrooms]             imputed 3 missing values

Final dataset size: 41,193 rows


---
## 4. Feature Engineering

### 4a. Host Category (`host_category`)
We classify hosts by how many total listings they manage on Airbnb (`calculated_host_listings_count`).  
The thresholds follow the literature on Airbnb professionalization (e.g., Wegmann & Jiao, 2017):

| `calculated_host_listings_count` | Category | Interpretation |
|---|---|---|
| 1 | **Casual** | True sharing-economy participant |
| 2–5 | **Semi-Pro** | Possibly an investment property, small-scale operator |
| > 5 | **Professional** | Commercial operator / property management company |

> **Note:** These thresholds are the *default* for this analysis. The JS component in Step 2 will allow readers to adjust the Professional threshold interactively.

### 4b. Revenue Proxy (`estimated_revenue_l365d`)
Inside Airbnb does not provide actual booking data. We approximate annual revenue as:

> `estimated_revenue = price × (365 − availability_365)`

This assumes that nights **not available** on the calendar were booked. It is a **conservative upper-bound proxy** — actual revenue may be lower due to cancellations and host blocks.


In [5]:
# ============================================================
# CELL 5 — FEATURE ENGINEERING
# ============================================================

# --- 5a. Host Category ---
def categorize_host(n):
    """Map listing count to host professionalization category."""
    if n == 1:
        return 'Casual'
    elif n <= 5:
        return 'Semi-Pro'
    else:
        return 'Professional'

df['host_category'] = df['calculated_host_listings_count'].apply(categorize_host)

print('Host category distribution:')
print(df.groupby('city')['host_category'].value_counts())

# --- 5b. Revenue Proxy ---
# Occupied nights = total nights minus nights still available
df['occupied_nights_proxy'] = (365 - df['availability_365']).clip(lower=0)
df['estimated_revenue_l365d'] = df['price'] * df['occupied_nights_proxy']

print('\nRevenue proxy summary (€/year):')
print(df.groupby('city')['estimated_revenue_l365d']
      .describe()[['mean', '50%', 'max']]
      .round(0))


Host category distribution:
city        host_category
Copenhagen  Casual           10042
            Semi-Pro          1060
            Professional       822
Rome        Casual           12005
            Semi-Pro          9417
            Professional      7847
Name: count, dtype: int64

Revenue proxy summary (€/year):
                mean       50%         max
city                                      
Copenhagen  293049.0  256637.0  33899661.0
Rome         24183.0   14700.0   3559644.0


---
## 5. Tourist Pressure — Neighbourhood Population Integration

To contextualize listing density, we calculate:

> `tourist_pressure = listings_in_neighbourhood / (population / 100)`
> (i.e., listings per 100 residents)

**Data source:** Neighbourhood population estimates are drawn from:
- **Rome:** Comune di Roma — Ufficio Statistico (2023 municipal registry)
- **Copenhagen:** Danmarks Statistik (2023 parish-level population)

Because a full administrative merge is outside the scope of this notebook, we embed a **representative sample of 15 high-tourism neighbourhoods per city** inline. The remaining neighbourhoods receive a `NaN` for `tourist_pressure`, which is treated as missing in Tableau.


In [6]:
# ============================================================
# CELL 6 — TOURIST PRESSURE (MOCK SECONDARY DATA)
# ============================================================

# Neighbourhood population estimates (source: city statistical offices, 2023)
# Format: {neighbourhood_cleansed_name: population}

ROME_POPULATION = {
    'Centro Storico':    15_000,
    'Trastevere':        12_000,
    'Prati':             35_000,
    'Testaccio':         14_000,
    'Pigneto':           20_000,
    'Ostiense':          18_000,
    'Nomentano':         45_000,
    'Trieste':           50_000,
    'Flaminio':          17_000,
    'Esquilino':         16_000,
    'San Giovanni':      30_000,
    'Monti':             10_000,
    'Aventino':           9_000,
    'Appio Latino':      55_000,
    'Parioli':           28_000,
}

COPENHAGEN_POPULATION = {
    'Indre By':          32_000,
    'Vesterbro-Kongens Enghave': 56_000,
    'Nørrebro':          78_000,
    'Frederiksberg':     40_000,
    'Østerbro':          69_000,
    'Bispebjerg':        56_000,
    'Amager Vest':       58_000,
    'Amager Øst':        39_000,
    'Valby':             43_000,
    'Brønshøj-Husum':    45_000,
    'Vanløse':           31_000,
    'Sundbyøster':       28_000,
    'Christianshavn':    12_000,
    'Islands Brygge':    16_000,
    'Sydhavn':           14_000,
}

# Build a unified lookup: (city, neighbourhood) → population
pop_rows = (
    [{'city': 'Rome',       'neighbourhood_cleansed': nb, 'population': pop}
     for nb, pop in ROME_POPULATION.items()] +
    [{'city': 'Copenhagen', 'neighbourhood_cleansed': nb, 'population': pop}
     for nb, pop in COPENHAGEN_POPULATION.items()]
)
pop_df = pd.DataFrame(pop_rows)

# Count listings per neighbourhood per city
listing_counts = (
    df.groupby(['city', 'neighbourhood_cleansed'])
    .size()
    .reset_index(name='listings_count')
)

# Merge population into listing counts
listing_counts = listing_counts.merge(pop_df, on=['city', 'neighbourhood_cleansed'], how='left')
listing_counts['tourist_pressure'] = (
    listing_counts['listings_count'] / (listing_counts['population'] / 100)
)

# Merge tourist pressure back into main dataframe
df = df.merge(
    listing_counts[['city', 'neighbourhood_cleansed', 'tourist_pressure', 'listings_count']],
    on=['city', 'neighbourhood_cleansed'],
    how='left'
)

covered = df['tourist_pressure'].notna().sum()
print(f'Tourist pressure available for {covered:,} listings ({covered/len(df)*100:.1f}%)')
print('\nTop 5 highest tourist-pressure neighbourhoods:')
print(
    listing_counts.dropna(subset=['tourist_pressure'])
    .nlargest(5, 'tourist_pressure')
    [['city', 'neighbourhood_cleansed', 'listings_count', 'tourist_pressure']]
    .round(2)
    .to_string(index=False)
)


Tourist pressure available for 7,220 listings (17.5%)

Top 5 highest tourist-pressure neighbourhoods:
      city    neighbourhood_cleansed  listings_count  tourist_pressure
Copenhagen                  Indre By            1934              6.04
Copenhagen Vesterbro-Kongens Enghave            2054              3.67
Copenhagen             Frederiksberg            1225              3.06
Copenhagen               Amager Vest            1065              1.84
Copenhagen                     Valby             414              0.96


---
## 6. EDA Summary
A brief statistical summary to validate the cleaned dataset before Tableau export.


In [7]:
# ============================================================
# CELL 7 — EDA SUMMARY
# ============================================================

print('=== DATASET OVERVIEW ===')
print(f'Total listings (after cleaning): {len(df):,}')
print(f'Unique hosts:                    {df["host_id"].nunique():,}')
print(f'Cities:                          {df["city"].unique()}')

print('\n=== HOST CATEGORY SHARES ===')
host_stats = (
    df.groupby(['city', 'host_category'])
    .agg(
        listings=('id', 'count'),
        total_revenue=('estimated_revenue_l365d', 'sum')
    )
    .reset_index()
)
for city in ['Rome', 'Copenhagen']:
    subset = host_stats[host_stats['city'] == city].copy()
    subset['listing_pct'] = subset['listings'] / subset['listings'].sum() * 100
    subset['revenue_pct'] = subset['total_revenue'] / subset['total_revenue'].sum() * 100
    print(f'\n{city}:')
    print(subset[['host_category', 'listings', 'listing_pct', 'revenue_pct']]
          .round(1)
          .to_string(index=False))

print('\n=== MEDIAN PRICE BY ROOM TYPE ===')
print(
    df.groupby(['city', 'room_type'])['price']
    .median()
    .reset_index()
    .rename(columns={'price': 'median_price_€'})
    .round(0)
    .to_string(index=False)
)


=== DATASET OVERVIEW ===
Total listings (after cleaning): 41,193
Unique hosts:                    27,519
Cities:                          <StringArray>
['Rome', 'Copenhagen']
Length: 2, dtype: str

=== HOST CATEGORY SHARES ===

Rome:
host_category  listings  listing_pct  revenue_pct
       Casual     12005         41.0         37.5
 Professional      7847         26.8         31.2
     Semi-Pro      9417         32.2         31.4

Copenhagen:
host_category  listings  listing_pct  revenue_pct
       Casual     10042         84.2         84.9
 Professional       822          6.9          7.5
     Semi-Pro      1060          8.9          7.6

=== MEDIAN PRICE BY ROOM TYPE ===
      city       room_type  median_price_€
Copenhagen Entire home/apt          1247.0
Copenhagen      Hotel room          1000.0
Copenhagen    Private room           588.0
Copenhagen     Shared room           224.0
      Rome Entire home/apt           140.0
      Rome      Hotel room           139.0
      Rome    Pri

---
## 7. Export to CSV
We select only the columns needed for Tableau, keeping the file small and readable.


In [8]:
# ============================================================
# CELL 8 — EXPORT CLEANED DATASET
# ============================================================

# Columns to include in the Tableau-ready export
EXPORT_COLS = [
    # Identifiers
    'id', 'city', 'name',
    # Location
    'neighbourhood_cleansed', 'latitude', 'longitude',
    # Listing attributes
    'room_type', 'bedrooms', 'bathrooms', 'accommodates',
    # Pricing
    'price', 'minimum_nights',
    # Availability & Revenue
    'availability_365', 'occupied_nights_proxy', 'estimated_revenue_l365d',
    # Host
    'host_id', 'host_since', 'host_is_superhost',
    'calculated_host_listings_count', 'host_category',
    # Reviews
    'number_of_reviews', 'review_scores_rating', 'reviews_per_month',
    # Tourist pressure (partial coverage)
    'tourist_pressure', 'listings_count',
]

# Keep only the columns that actually exist in df (handles minor schema differences per city)
export_cols_available = [c for c in EXPORT_COLS if c in df.columns]
export_df = df[export_cols_available].copy()

# Save
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
export_df.to_csv(OUTPUT_CSV, index=False)

print(f'Exported {len(export_df):,} rows × {len(export_df.columns)} columns')
print(f'Saved to: {OUTPUT_CSV}')
print(f'File size: {OUTPUT_CSV.stat().st_size / 1_048_576:.1f} MB')
print('\nFirst 3 rows:')
export_df.head(3)


Exported 41,193 rows × 25 columns
Saved to: data/airbnb_cleaned_final.csv
File size: 7.9 MB

First 3 rows:


,id,city,name,neighbourhood_cleansed,latitude,longitude,room_type,bedrooms,bathrooms,accommodates,price,minimum_nights,availability_365,occupied_nights_proxy,estimated_revenue_l365d,host_id,host_since,host_is_superhost,calculated_host_listings_count,host_category,number_of_reviews,review_scores_rating,reviews_per_month,tourist_pressure,listings_count
0,2737,Rome,"Elif's room in cozy, clean flat.",VIII Appia Antica,41.871360,12.482150,Private room,1.0,1.5,1,57.0,31,365,0,0.0,3047,2008-09-18,f,6,Professional,5,4.80,0.04,NaN,942
1,11834,Rome,"Charming Boschetto Studio, Rome",I Centro Storico,41.895447,12.491181,Entire home/apt,1.0,1.0,2,110.0,2,295,70,7700.0,44552,2009-10-09,t,1,Casual,284,4.86,1.62,NaN,14723
2,12398,Rome,Casa Donatello - Home far from Home,II Parioli/Nomentano,41.925820,12.469280,Entire home/apt,2.0,1.0,6,124.0,3,162,203,25172.0,11756,2009-03-30,t,1,Casual,85,4.90,0.47,NaN,2002
